# Exploratory Data Analysis (EDA) - House Price Prediction
This notebook performs a comprehensive Exploratory Data Analysis on the Kaggle House Prices dataset (Ames Housing).
It covers data loading, target distribution analysis, null value checking, duplicate identification, and key visual relationships.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set aesthetic parameters for visualization
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.titlesize'] = 16

## 1. Data Collection & Inspection
We start by downloading and loading the datasets using the modular loader functions.

In [ ]:
import sys
sys.path.append('../')
from src.data_loader import download_datasets, load_data

# Ensure dataset is downloaded and load it
download_datasets(dest_dir="../dataset")
train_df, test_df = load_data(data_dir="../dataset")

In [ ]:
print(f"Train set shape: {train_df.shape}")
print(f"Test set shape: {test_df.shape}")

In [ ]:
# Preview dataset columns
train_df.head()

In [ ]:
# Verify dataset info (dtypes, non-null counts)
train_df.info()

In [ ]:
# Numerical features statistical summary
train_df.describe().T.head(15)

## 2. Null Value and Duplicate Analysis
Let's identify columns with missing data and check for duplicate rows.

In [ ]:
# Duplicate check
duplicates = train_df.duplicated().sum()
print(f"Number of duplicate records: {duplicates}")

In [ ]:
# Missing value percentage per column
missing = train_df.isnull().sum()
missing_pct = 100 * missing / len(train_df)
missing_df = pd.DataFrame({'Missing Count': missing, 'Percentage': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values(by='Percentage', ascending=False)
missing_df.head(20)

In [ ]:
# Visualize top missing values
if not missing_df.empty:
    plt.figure(figsize=(12, 6))
    sns.barplot(x=missing_df.index[:15], y=missing_df['Percentage'][:15], palette='flare')
    plt.xticks(rotation=45)
    plt.ylabel('Percentage of Missing Values (%)')
    plt.title('Top 15 Columns with Missing Values')
    plt.tight_layout()
    plt.show()

## 3. Target Variable Analysis: SalePrice
Analyzing `SalePrice` is critical as it is the target we wish to predict. House prices usually present a right-skewed distribution, which violates the normality assumption of regression algorithms.

In [ ]:
# Target variable distribution (original scale)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.histplot(train_df['SalePrice'], kde=True, color='blue', ax=axes[0])
axes[0].set_title('Distribution of SalePrice (Original Scale)')
axes[0].set_xlabel('SalePrice ($)')

# Target variable distribution (log scale)
sns.histplot(np.log1p(train_df['SalePrice']), kde=True, color='green', ax=axes[1])
axes[1].set_title('Distribution of SalePrice (Log scale - log1p)')
axes[1].set_xlabel('log(SalePrice + 1)')

plt.tight_layout()
plt.show()

## 4. Correlation Analysis
Let's identify numerical features that correlate strongly with the target.

In [ ]:
# Compute correlation matrix
numerical_train = train_df.select_dtypes(exclude=['object'])
corr_matrix = numerical_train.corr()

# Top correlations with SalePrice
top_corrs = corr_matrix['SalePrice'].sort_values(ascending=False).head(15)
print("Top 15 features correlated with SalePrice:\n", top_corrs)

In [ ]:
# Heatmap of top correlated features
top_features = top_corrs.index.tolist()
plt.figure(figsize=(12, 10))
sns.heatmap(numerical_train[top_features].corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Matrix Heatmap of Top Features')
plt.tight_layout()
plt.show()

## 5. Visualizing Key Relationships
Let's create scatter plots of continuous variables (`GrLivArea`, `TotalBsmtSF`) vs `SalePrice`, and box plots of categorical ratings (`OverallQual`) vs `SalePrice`.

In [ ]:
# Scatter plot: GrLivArea vs SalePrice
plt.figure(figsize=(10, 6))
sns.scatterplot(data=train_df, x='GrLivArea', y='SalePrice', alpha=0.7, color='teal')
plt.title('GrLivArea vs SalePrice (Identify Outliers)')
plt.xlabel('GrLivArea (Above Ground Living Area sq ft)')
plt.ylabel('SalePrice ($)')
plt.show()

In [ ]:
# Box plot: OverallQual vs SalePrice
plt.figure(figsize=(10, 6))
sns.boxplot(data=train_df, x='OverallQual', y='SalePrice', palette='viridis')
plt.title('Overall Quality vs SalePrice')
plt.xlabel('Overall Quality Rating (1-10)')
plt.ylabel('SalePrice ($)')
plt.show()